# 🎤 KaraokeAI Backend Server
### Run this notebook ONCE. Future updates auto-apply via git pull!
**Steps:** Click **Runtime → Run all** after setting your ngrok token in Cell 3.

In [ ]:
# -- CELL 1: Install dependencies --
print('Installing dependencies...')

import subprocess, sys

# Install demucs (standard install to pull sphn and all sub-dependencies)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'demucs', 'sphn'], check=False)

# Install remaining lightweight packages
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'faster-whisper', 'ffmpeg-python', 'pysubs2',
    'fastapi', 'uvicorn', 'pyngrok', 'nest-asyncio',
    'transliterate', 'anyascii', 'aksharamukha',
    'fastapi[standard]', 'yt-dlp'
], check=False)

# ffmpeg system binary (usually already present on Colab)
subprocess.run(['apt-get', 'install', '-q', '-y', 'ffmpeg'], capture_output=True)

print('All dependencies installed!')


In [ ]:
# ── CELL 2: Clone/update your GitHub repository ───────────────
# api_server.py lives in the repo now — git pull gets the latest automatically!
import os

GITHUB_REPO = 'https://github.com/shanmuganathan457/karaoke-video-generator.git'

if os.path.exists('karaoke-video-generator'):
    print('Repo already exists, pulling latest changes...')
    os.system('cd karaoke-video-generator && git pull')
else:
    print('Cloning your GitHub repository...')
    os.system(f'git clone {GITHUB_REPO}')

os.chdir('karaoke-video-generator')
print(f'✅ Working directory: {os.getcwd()}')
print('📁 api_server.py exists:', os.path.exists('api_server.py'))

In [ ]:
# ── CELL 3: Enter your ngrok token ────────────────────────────
# Get your FREE token from: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('✅ ngrok token set!')

In [ ]:
# -- CELL 4: START THE SERVER --
import subprocess, time, sys, requests
from pyngrok import ngrok

# Step 1: Kill port 8000 (fixes 'address already in use')
subprocess.run(['fuser', '-k', '8000/tcp'], capture_output=True)
time.sleep(1)
print('Port 8000 cleared.')

# Step 2: Kill tunnels via ngrok local REST API (most reliable method)
# This deletes the cloud endpoint, not just the local process
try:
    resp = requests.get('http://localhost:4040/api/tunnels', timeout=3)
    for t in resp.json().get('tunnels', []):
        requests.delete(f"http://localhost:4040/api/tunnels/{t['name']}", timeout=3)
        print(f'Killed cloud tunnel: {t["public_url"]}')
except Exception as e:
    print(f'No local tunnels to kill via API: {e}')

# Step 3: Also kill the ngrok process itself as a backup
subprocess.run(['pkill', '-f', 'ngrok'], capture_output=True)
time.sleep(3)
print('ngrok process killed.')

# Step 4: Start uvicorn - logs to file so we can debug if it crashes
log = open('/tmp/uvicorn.log', 'w')
proc = subprocess.Popen(
    ['uvicorn', 'api_server:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=log, stderr=log
)
time.sleep(5)

if proc.poll() is not None:
    print('Server CRASHED! Error:')
    print(open('/tmp/uvicorn.log').read())
    raise RuntimeError('uvicorn failed to start. See error above.')

# Step 5: Connect ngrok (set auth token first if needed)
from pyngrok import ngrok
public_url = ngrok.connect(8000)

print('=' * 60)
print('KaraokeAI Backend is LIVE!')
print('=' * 60)
print(f'YOUR PUBLIC API URL: {public_url}')
print('Copy this URL and paste it into your app!')
print('Keep this tab open - closing it stops the server.')
print('=' * 60)

while True:
    time.sleep(60)
    lines = open('/tmp/uvicorn.log').readlines()
    last = lines[-1].strip() if lines else '...'
    print(f'Server still running... ({last})')
